# SLTFM · Llama 3 8B continual pretraining on Sri Lankan Tamil (Kaggle, 2× T4)

Parameter-efficient continual pretraining (QLoRA) of **Meta Llama 3 8B Base** on
[`ctnlpr/sri-lankan-tamil-corpus`](https://huggingface.co/datasets/ctnlpr/sri-lankan-tamil-corpus) (`documents` config),
using both T4 GPUs with DDP.

**Before the first run**
1. *Settings → Accelerator:* **GPU T4 ×2** · *Settings → Internet:* **On**
2. *Add-ons → Secrets:* add **`HF_TOKEN`** (a *write* token) and **`WANDB_API_KEY`**, and tick both for this notebook.
3. With the same HF account, accept the license at https://huggingface.co/meta-llama/Meta-Llama-3-8B (gated model).

**How to use**
- Keep `SMOKE_TEST = True` → *Run All*. It trains a few steps, pushes a checkpoint, simulates a crash, wipes the local disk,
  resumes from the Hub checkpoint, finishes, pushes the final adapter, runs a generation test and prints an ETA for the full run.
- When every check is ✅, set `SMOKE_TEST = False` and use **Save Version → Save & Run All (Commit)** so it runs in the background.
- **Resuming:** just run the notebook again. It finds the newest `checkpoint-N` on the Hub and continues from it
  (weights, optimizer, scheduler, RNG and data position). Do not change hyper-parameters between resumes.

## 1 · Install

In [ ]:
!pip install -q -U "transformers>=4.51,<4.57" "peft>=0.15,<0.18" "accelerate>=1.6,<2" "bitsandbytes>=0.45" "datasets>=3.2" wandb
!python -c "import torch, transformers, peft, accelerate, bitsandbytes, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| accelerate', accelerate.__version__, '| bitsandbytes', bitsandbytes.__version__, '| datasets', datasets.__version__)"

## 2 · Configuration

In [ ]:
# ======================= MAIN SWITCH =======================
SMOKE_TEST = True     # True  = quick end-to-end test (incl. crash + resume from Hub)
                      # False = full training on the whole corpus
# ===========================================================

BASE_MODEL     = "meta-llama/Meta-Llama-3-8B"          # primary model in the SLTFM plan
DATASET_ID     = "ctnlpr/sri-lankan-tamil-corpus"
DATASET_CONFIG = "documents"                           # one row = one cleaned document (for CPT)
HUB_REPO_NAME  = "Llama-3-SLTFM-8B-CPT-LoRA"           # private repo under your HF account ("-smoke" added in smoke mode)
WANDB_PROJECT  = "sltfm-cpt"

# --- model / LoRA ---
SEQ_LEN      = 2048
LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- optimisation (full run) ---
PER_DEVICE_BATCH = 1
GRAD_ACCUM       = 8          # global batch = 1 x 8 x 2 GPUs x 2048 = 32,768 tokens / step
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 2
WARMUP_RATIO     = 0.05
SAVE_STEPS       = 25         # every checkpoint is pushed to the Hub
EVAL_STEPS       = 25
LOGGING_STEPS    = 1
MAX_EVAL_BLOCKS  = 100        # fixed validation subset (blocks of SEQ_LEN tokens) to keep eval fast
MAX_RUNTIME_HOURS = 10.5      # clean save + push + stop before Kaggle's 12 h session limit
SEED = 42

# --- smoke-test overrides ---
SMOKE_CFG = dict(max_steps=10, crash_at_step=5, save_steps=5, eval_steps=5,
                 grad_accum=2, train_blocks=64, eval_blocks=8)

# --- paths (derived) ---
import os
MODE       = "smoke" if SMOKE_TEST else "full"
WORK_DIR   = "/kaggle/working"
SCRIPT_DIR = f"{WORK_DIR}/sltfm"
OUTPUT_DIR = f"{WORK_DIR}/ckpt_{MODE}"
DATA_DIR   = f"/tmp/sltfm_data_{MODE}_seq{SEQ_LEN}"
CFG_PATH   = f"{SCRIPT_DIR}/config_{MODE}.json"
RUN_NAME   = f"llama3-8b-sltfm-cpt-r{LORA_R}" + ("-smoke" if SMOKE_TEST else "")
os.makedirs(SCRIPT_DIR, exist_ok=True)
print("MODE:", MODE)

## 3 · Secrets, GPU check, Hugging Face access

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
HF_TOKEN = _secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = _secrets.get_secret("WANDB_API_KEY")

os.environ.update({
    "HF_TOKEN": HF_TOKEN,
    "WANDB_API_KEY": WANDB_API_KEY,
    "HF_HOME": "/tmp/hf_home",                 # big base-model download stays out of /kaggle/working
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "WANDB_LOG_MODEL": "false",
    "WANDB_WATCH": "false",
})

from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()["name"]
REPO_ID = f"{HF_USER}/{HUB_REPO_NAME}" + ("-smoke" if SMOKE_TEST else "")

gpus = subprocess.check_output(["nvidia-smi", "-L"], text=True).strip().splitlines()
N_GPUS = len(gpus)
print("\n".join(gpus))
if N_GPUS < 2:
    print(f"WARNING: expected 2 GPUs (T4 x2) but found {N_GPUS}; training will use {N_GPUS}.")

try:
    hf_hub_download(BASE_MODEL, "config.json", token=HF_TOKEN)
    print(f"\nAccess to {BASE_MODEL}: OK")
except Exception as e:
    raise RuntimeError(f"No access to {BASE_MODEL}. Accept the Llama 3 license on its HF page with this account.\n{e}")

api.create_repo(REPO_ID, private=True, exist_ok=True)   # also verifies the token has write access
print(f"HF user        : {HF_USER}\nCheckpoint repo: https://huggingface.co/{REPO_ID}")

## 4 · Scripts
Training runs as a separate `torchrun` process (one process per GPU), so the notebook kernel never holds GPU memory.

In [ ]:
%%writefile /kaggle/working/sltfm/prepare_data.py
"""Tokenize + pack the Sri Lankan Tamil corpus into fixed-length blocks for CPT."""
import argparse
import json
import os
import re
import shutil
import unicodedata

from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
cfg = json.load(open(ap.parse_args().config))
SEQ = cfg["seq_len"]

tok = AutoTokenizer.from_pretrained(cfg["base_model"])
tok.model_max_length = int(1e9)  # we chunk ourselves; silences the length warning
EOS = tok.eos_token_id
print(f"tokenizer: vocab={len(tok)} | bos={tok.bos_token} | eos={tok.eos_token}")

raw = load_dataset(cfg["dataset_id"], cfg["dataset_config"], token=os.environ.get("HF_TOKEN"))
print(raw)

_ws = re.compile(r"[ \t\u00a0]+")


def clean(t):
    t = unicodedata.normalize("NFC", t or "")
    t = _ws.sub(" ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def tokenize(batch):
    texts = [clean(t) for t in batch["text"]]
    ids = tok(texts, add_special_tokens=True)["input_ids"]  # prepends <|begin_of_text|>
    return {"input_ids": [x + [EOS] for x in ids],             # appends <|end_of_text|>
            "n_chars": [len(t) for t in texts],
            "n_words": [len(t.split()) for t in texts]}


stats, out = {}, {}
for split in raw:
    d = raw[split].filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)
    d = d.map(tokenize, batched=True, batch_size=8, remove_columns=d.column_names, desc=f"tokenize {split}")
    flat = [i for seq in d["input_ids"] for i in seq]
    n_blocks = len(flat) // SEQ
    n_chars, n_words, n_tok = sum(d["n_chars"]), sum(d["n_words"]), len(flat)
    stats[split] = {
        "documents": len(d), "chars": n_chars, "words": n_words, "tokens": n_tok,
        "chars_per_token": round(n_chars / max(n_tok, 1), 3),
        "tokens_per_word (fertility)": round(n_tok / max(n_words, 1), 3),
        "blocks": n_blocks, "dropped_tail_tokens": n_tok - n_blocks * SEQ,
    }
    out[split] = Dataset.from_dict({"input_ids": [flat[k * SEQ:(k + 1) * SEQ] for k in range(n_blocks)]})

limits = {"train": cfg.get("max_train_blocks"), "validation": cfg.get("max_eval_blocks")}
for split, n in limits.items():
    if split in out and n and len(out[split]) > n:
        out[split] = out[split].shuffle(seed=cfg["seed"]).select(range(n))  # fixed subset across sessions
for split in out:
    stats[split]["blocks_used"] = len(out[split])

shutil.rmtree(cfg["data_dir"], ignore_errors=True)
DatasetDict(out).save_to_disk(cfg["data_dir"])
with open(cfg["data_dir"] + "_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

In [ ]:
%%writefile /kaggle/working/sltfm/train.py
"""SLTFM: QLoRA continual pretraining of Llama 3 8B on Sri Lankan Tamil (DDP on 2x T4).

Launched by the notebook with torchrun. Every saved checkpoint (adapter + optimizer +
scheduler + RNG + trainer state) is uploaded synchronously to the Hub, so a run can be
resumed from the newest checkpoint even after the whole Kaggle session is lost.
"""
import argparse
import json
import math
import os
import statistics
import time

import torch
import torch.distributed as dist
from datasets import load_from_disk
from huggingface_hub import HfApi
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          Trainer, TrainerCallback, TrainingArguments, set_seed)

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
cfg = json.load(open(ap.parse_args().config))

TOKEN = os.environ["HF_TOKEN"]
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", 0))
RANK = int(os.environ.get("RANK", 0))
WORLD = int(os.environ.get("WORLD_SIZE", 1))
IS_MAIN = RANK == 0
torch.cuda.set_device(LOCAL_RANK)


def log(*a):
    if IS_MAIN:
        print("[train]", *a, flush=True)


def dist_on():
    return dist.is_available() and dist.is_initialized()


def barrier():
    if dist_on():
        dist.barrier()


def upload_with_retry(api, attempts=5, **kw):
    for i in range(1, attempts + 1):
        try:
            return api.upload_folder(**kw)
        except Exception as e:  # network hiccups on Kaggle are common
            print(f"[train] upload attempt {i}/{attempts} failed: {e}", flush=True)
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status is not None and 400 <= status < 500 and status != 429:
                print("[train] permanent error (HTTP %s): not retrying" % status, flush=True)
                return None
            time.sleep(min(90, 20 * i))
    return None


# --------------------------------------------------------------------------- callbacks
class HubCheckpointPusher(TrainerCallback):
    """Uploads every checkpoint-N folder to the Hub as one atomic commit (rank 0 only)."""

    def __init__(self, repo_id):
        self.api = HfApi(token=TOKEN)
        self.repo_id = repo_id

    def on_save(self, args, state, control, **kwargs):
        barrier()  # every rank has finished writing (e.g. rng_state_1.pth)
        if state.is_world_process_zero:
            name = f"checkpoint-{state.global_step}"
            path = os.path.join(args.output_dir, name)
            t0 = time.time()
            # PEFT writes a README.md whose base_model is the local snapshot path; the Hub
            # rejects that metadata and the whole commit fails. Resuming does not need it.
            ok = upload_with_retry(self.api, repo_id=self.repo_id, folder_path=path, path_in_repo=name,
                                   ignore_patterns=["README.md"],
                                   commit_message=f"{name} (epoch {state.epoch:.3f})")
            if ok is not None:
                log(f"pushed {name} to {self.repo_id} in {time.time() - t0:.0f}s")
            else:
                log(f"WARNING: {name} could not be pushed; it is still on local disk at {path}")
        barrier()
        return control


class RunController(TrainerCallback):
    """Clean stop (save + push) when the runtime budget is used up, or at crash_at_step (smoke test)."""

    def __init__(self, max_hours, crash_at_step=0):
        self.t0 = time.time()
        self.budget = max_hours * 3600
        self.crash_at = crash_at_step or 0
        self.reason = None
        self.start_step = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_step = state.global_step  # already restored from the checkpoint when resuming

    def on_step_end(self, args, state, control, **kwargs):
        flag = 0
        if state.is_world_process_zero:
            if self.crash_at and state.global_step >= self.crash_at:
                flag = 1
            elif time.time() - self.t0 > self.budget:
                flag = 2
        if dist_on():  # all ranks must take the same decision, otherwise DDP hangs
            t = torch.tensor([flag], device="cuda")
            dist.broadcast(t, src=0)
            flag = int(t.item())
        if flag:
            self.reason = "simulated_crash" if flag == 1 else "time_budget"
            log(f"stopping at step {state.global_step}: {self.reason} (checkpoint will be saved + pushed)")
            control.should_save = True
            control.should_training_stop = True
        return control


class CPTTrainer(Trainer):
    """Adds perplexity, tokens/sec and ETA to every log line (console + W&B)."""

    def __init__(self, *args, tokens_per_step=0, **kwargs):
        self.tokens_per_step = tokens_per_step
        self._tp_t = None
        self._tp_step = None
        self.tps_samples = []
        super().__init__(*args, **kwargs)

    def log(self, logs, *args, **kwargs):
        if "eval_loss" in logs:
            logs["eval_perplexity"] = math.exp(min(logs["eval_loss"], 30))
        if "loss" in logs:
            logs["train_perplexity"] = math.exp(min(logs["loss"], 30))
            now, step = time.time(), self.state.global_step
            if self._tp_t is not None and step > self._tp_step:
                tps = (step - self._tp_step) * self.tokens_per_step / (now - self._tp_t)
                self.tps_samples.append(tps)
                logs["tokens_per_sec"] = round(tps, 1)
                logs["eta_hours"] = round((self.state.max_steps - step) * self.tokens_per_step / tps / 3600, 2)
            self._tp_t, self._tp_step = now, step
        super().log(logs, *args, **kwargs)


def collate(features):
    ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
    return {"input_ids": ids, "attention_mask": torch.ones_like(ids), "labels": ids.clone()}


def model_card(status):
    ev = status.get("final_eval", {})
    hist = status.get("eval_history", [])
    base_ppl = f"{math.exp(hist[0][1]):.2f}" if hist and hist[0][0] == 0 else "n/a"
    final_ppl = f"{math.exp(ev['eval_loss']):.2f}" if "eval_loss" in ev else "n/a"
    return f"""---
base_model: {cfg['base_model_id']}
library_name: peft
license: llama3
language:
- ta
datasets:
- {cfg['dataset_id']}
tags:
- llama-3
- qlora
- continual-pretraining
- sri-lankan-tamil
- sltfm
---

# {cfg['repo_id'].split('/')[-1]}

Sri Lankan Tamil Foundation Model (SLTFM) - LoRA adapter from parameter-efficient continual
pretraining (QLoRA, 4-bit NF4 base) of `{cfg['base_model_id']}` on `{cfg['dataset_id']}`
(config `{cfg['dataset_config']}`). Built with Meta Llama 3.

| Setting | Value |
|---|---|
| LoRA r / alpha / dropout | {cfg['lora_r']} / {cfg['lora_alpha']} / {cfg['lora_dropout']} |
| Target modules | {', '.join(cfg['lora_targets'])} |
| Sequence length | {cfg['seq_len']} (packed) |
| Tokens per optimizer step | {status.get('tokens_per_step')} |
| Steps / epochs | {status.get('end_step')} / {cfg['num_epochs']} |
| Learning rate | {cfg['lr']} (cosine, warmup ratio {cfg['warmup_ratio']}) |
| Tokenizer | original Llama 3 tokenizer (no Tamil vocabulary expansion) |
| Validation perplexity: base Llama 3 | {base_ppl} |
| Validation perplexity: after CPT | {final_ppl} |

Every intermediate checkpoint (with optimizer state) is kept in the `checkpoint-*` folders.

## Usage

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("{cfg['base_model_id']}", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, "{cfg['repo_id']}")
tok = AutoTokenizer.from_pretrained("{cfg['repo_id']}")
```
"""


def push_final(trainer, tok, status):
    final_dir = os.path.join(cfg["output_dir"], "final_adapter")
    trainer.model.save_pretrained(final_dir)
    tok.save_pretrained(final_dir)
    with open(os.path.join(final_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(model_card(status))
    with open(os.path.join(final_dir, "training_complete.json"), "w") as f:
        json.dump(status, f, indent=2)
    ok = upload_with_retry(HfApi(token=TOKEN), repo_id=cfg["repo_id"], folder_path=final_dir,
                           commit_message="Final SLTFM CPT adapter")
    log("final adapter pushed to repo root" if ok is not None else "WARNING: final adapter push FAILED")


# --------------------------------------------------------------------------- main
def main():
    set_seed(cfg["seed"])
    tok = AutoTokenizer.from_pretrained(cfg["base_model"])
    data = load_from_disk(cfg["data_dir"])
    train_ds, eval_ds = data["train"], data["validation"]
    log(f"world_size={WORLD} | train blocks={len(train_ds)} | eval blocks={len(eval_ds)} | seq_len={cfg['seq_len']}")

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)  # T4 has no bf16
    model = AutoModelForCausalLM.from_pretrained(
        cfg["base_model"], quantization_config=bnb, torch_dtype=torch.float16,
        device_map={"": LOCAL_RANK}, attn_implementation="sdpa")
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["lora_targets"], bias="none", task_type="CAUSAL_LM"))
    # adapter_config.json should point at the Hub id, not the local snapshot path
    for pc in model.peft_config.values():
        pc.base_model_name_or_path = cfg["base_model_id"]
    if IS_MAIN:
        model.print_trainable_parameters()

    tokens_per_step = cfg["per_device_batch"] * cfg["grad_accum"] * WORLD * cfg["seq_len"]
    log(f"tokens per optimizer step: {tokens_per_step:,}")

    args = TrainingArguments(
        output_dir=cfg["output_dir"],
        run_name=cfg["run_name"],
        per_device_train_batch_size=cfg["per_device_batch"],
        per_device_eval_batch_size=cfg["per_device_batch"],
        gradient_accumulation_steps=cfg["grad_accum"],
        learning_rate=cfg["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=cfg["warmup_ratio"],
        weight_decay=0.0,
        max_grad_norm=1.0,
        num_train_epochs=cfg["num_epochs"],
        max_steps=cfg["max_steps"],
        fp16=True,
        bf16=False,
        optim="adamw_bnb_8bit",  # NOT paged: paged 8-bit optimizers crash when resuming from a checkpoint
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=cfg["logging_steps"],
        eval_strategy="steps",
        eval_steps=cfg["eval_steps"],
        eval_on_start=cfg["eval_on_start"],
        save_strategy="steps",
        save_steps=cfg["save_steps"],
        save_total_limit=2,  # local disk only; every checkpoint is kept on the Hub
        report_to=["wandb"],
        seed=cfg["seed"],
        data_seed=cfg["seed"],
        dataloader_num_workers=2,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
        ddp_timeout=10800,
        disable_tqdm=True,
    )

    controller = RunController(cfg["max_runtime_hours"], cfg.get("crash_at_step", 0))
    trainer = CPTTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collate,
        callbacks=[HubCheckpointPusher(cfg["repo_id"]), controller], tokens_per_step=tokens_per_step)

    resume = cfg.get("resume_from")
    log(f"RESUMING from {resume}" if resume else "Fresh start (no checkpoint found)")
    trainer.train(resume_from_checkpoint=resume)

    st = trainer.state
    finished = controller.reason is None and st.global_step >= st.max_steps
    status = {
        "finished": finished,
        "stopped_reason": controller.reason,
        "start_step": controller.start_step,
        "end_step": st.global_step,
        "max_steps": st.max_steps,
        "epoch": st.epoch,
        "tokens_per_step": tokens_per_step,
        "world_size": WORLD,
        "tps_samples": trainer.tps_samples,
        "median_tokens_per_sec": statistics.median(trainer.tps_samples) if trainer.tps_samples else None,
        "eval_history": [(h["step"], h["eval_loss"]) for h in st.log_history if "eval_loss" in h],
    }
    if finished:
        status["final_eval"] = trainer.evaluate()
        log(f"final eval: {status['final_eval']}")
        if IS_MAIN:
            push_final(trainer, tok, status)
        barrier()
    if IS_MAIN:
        with open(os.path.join(cfg["output_dir"], "run_status.json"), "w") as f:
            json.dump(status, f, indent=2)
        try:
            import wandb
            if wandb.run is not None:
                wandb.finish()
        except Exception:
            pass
    barrier()


if __name__ == "__main__":
    main()
    if dist_on():
        dist.destroy_process_group()

In [ ]:
%%writefile /kaggle/working/sltfm/generate.py
"""Qualitative check: base Llama 3 vs. Llama 3 + SLTFM adapter on Sri Lankan Tamil prompts."""
import argparse
import json
import os

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True, help="HF repo id holding the adapter")
ap.add_argument("--subfolder", default="", help="e.g. checkpoint-100; empty = final adapter at repo root")
ap.add_argument("--max_new_tokens", type=int, default=120)
a = ap.parse_args()
cfg = json.load(open(a.config))

PROMPTS = [
    "யாழ்ப்பாணம் இலங்கையின் வடக்கே அமைந்துள்ள",
    "ஈழத்துத் தமிழ் இலக்கிய வரலாற்றில்",
    "இலங்கையில் கூட்டுறவு இயக்கம்",
]

tok = AutoTokenizer.from_pretrained(cfg["base_model"])
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(cfg["base_model"], quantization_config=bnb,
                                             torch_dtype=torch.float16, device_map={"": 0})
kw = {"subfolder": a.subfolder} if a.subfolder else {}
model = PeftModel.from_pretrained(model, a.adapter, token=os.environ["HF_TOKEN"], **kw)
model.eval()


def generate(prompt):
    x = tok(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=a.max_new_tokens, do_sample=False,
                           repetition_penalty=1.15, pad_token_id=tok.eos_token_id)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)


for p in PROMPTS:
    print("=" * 90)
    print("PROMPT:", p)
    with model.disable_adapter():
        print("\n[base Llama 3 8B]\n" + generate(p))
    print("\n[Llama 3 8B + SLTFM adapter]\n" + generate(p))
print("=" * 90)

## 5 · Helpers: config, Hub checkpoint discovery, resume, launcher

In [ ]:
import glob, json, math, re, shutil, statistics, subprocess, sys, uuid
from huggingface_hub import snapshot_download


def run_streaming(cmd, extra_env=None):
    """Run a subprocess and stream its output into the notebook."""
    env = os.environ.copy()
    env.update(extra_env or {})
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def build_config(resume_from=None, crash_at_step=0):
    s = SMOKE_TEST
    return dict(
        base_model=BASE_LOCAL, base_model_id=BASE_MODEL,
        dataset_id=DATASET_ID, dataset_config=DATASET_CONFIG,
        data_dir=DATA_DIR, output_dir=OUTPUT_DIR, repo_id=REPO_ID, run_name=RUN_NAME,
        seq_len=SEQ_LEN, seed=SEED,
        lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, lora_targets=LORA_TARGETS,
        per_device_batch=PER_DEVICE_BATCH,
        grad_accum=SMOKE_CFG["grad_accum"] if s else GRAD_ACCUM,
        lr=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, num_epochs=NUM_EPOCHS,
        max_steps=SMOKE_CFG["max_steps"] if s else -1,
        save_steps=SMOKE_CFG["save_steps"] if s else SAVE_STEPS,
        eval_steps=SMOKE_CFG["eval_steps"] if s else EVAL_STEPS,
        logging_steps=1 if s else LOGGING_STEPS,
        max_train_blocks=SMOKE_CFG["train_blocks"] if s else None,
        max_eval_blocks=SMOKE_CFG["eval_blocks"] if s else MAX_EVAL_BLOCKS,
        max_runtime_hours=MAX_RUNTIME_HOURS,
        resume_from=resume_from,
        eval_on_start=resume_from is None,     # baseline perplexity of plain Llama 3 on a fresh start
        crash_at_step=crash_at_step,
    )


def hub_files(repo_id):
    try:
        return set(api.list_repo_files(repo_id))
    except Exception:
        return set()


def training_complete(repo_id):
    return "training_complete.json" in hub_files(repo_id)


_CKPT_RE = re.compile(r"^checkpoint-(\d+)/trainer_state\.json$")

def hub_checkpoint_steps(repo_id):
    # each checkpoint is uploaded as one atomic commit, so trainer_state.json present = complete checkpoint
    return sorted(int(m.group(1)) for f in hub_files(repo_id) if (m := _CKPT_RE.match(f)))


def local_checkpoint_steps():
    return sorted(int(d.rsplit("-", 1)[1]) for d in glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
                  if os.path.exists(f"{d}/trainer_state.json"))


def resolve_resume():
    hub, loc = hub_checkpoint_steps(REPO_ID), local_checkpoint_steps()
    h, l = (hub[-1] if hub else -1), (loc[-1] if loc else -1)
    print(f"Hub checkpoints: {hub or 'none'} | local checkpoints: {loc or 'none'}")
    if h < 0 and l < 0:
        return None
    if l >= h:
        print(f"-> resuming from LOCAL checkpoint-{l}")
        return f"{OUTPUT_DIR}/checkpoint-{l}"
    name = f"checkpoint-{h}"
    print(f"-> downloading {name} from the Hub ...")
    snapshot_download(REPO_ID, allow_patterns=[f"{name}/*"], local_dir=OUTPUT_DIR, token=HF_TOKEN)
    return f"{OUTPUT_DIR}/{name}"


def get_wandb_run_id():
    """Same W&B run across all resumes: the id is stored in the Hub repo."""
    if "wandb_run_id.txt" in hub_files(REPO_ID):
        with open(hf_hub_download(REPO_ID, "wandb_run_id.txt", token=HF_TOKEN, force_download=True)) as f:
            return f.read().strip()
    rid = uuid.uuid4().hex[:10]
    api.upload_file(path_or_fileobj=rid.encode(), path_in_repo="wandb_run_id.txt", repo_id=REPO_ID,
                    commit_message="W&B run id")
    return rid


def launch_training(crash_at_step=0):
    api.create_repo(REPO_ID, private=True, exist_ok=True)
    if training_complete(REPO_ID):
        print(f"{REPO_ID} already holds a finished run (training_complete.json). Nothing to do.")
        return {"finished": True, "already_complete": True}
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    resume = resolve_resume()
    cfg = build_config(resume_from=resume, crash_at_step=crash_at_step)
    with open(CFG_PATH, "w") as f:
        json.dump(cfg, f, indent=2)
    api.upload_file(path_or_fileobj=CFG_PATH, path_in_repo="sltfm_run_config.json", repo_id=REPO_ID,
                    commit_message="Run config")
    status_path = f"{OUTPUT_DIR}/run_status.json"
    if os.path.exists(status_path):
        os.remove(status_path)
    env = {"WANDB_PROJECT": WANDB_PROJECT, "WANDB_RUN_ID": get_wandb_run_id(),
           "WANDB_RESUME": "allow", "WANDB_NAME": RUN_NAME}
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={N_GPUS}",
           f"{SCRIPT_DIR}/train.py", "--config", CFG_PATH]
    rc = run_streaming(cmd, env)
    if rc != 0:
        raise RuntimeError(f"Training exited with code {rc}. Fix the error above and re-run this cell: "
                           "it resumes from the newest checkpoint on the Hub.")
    with open(status_path) as f:
        return json.load(f)


print("helpers ready")

## 6 · Download Llama 3 8B Base (safetensors only)

In [ ]:
BASE_LOCAL = snapshot_download(
    BASE_MODEL, token=HF_TOKEN,
    allow_patterns=["*.json", "*.safetensors", "tokenizer*"],   # skips the duplicate original/*.pth weights
)
print("base model at:", BASE_LOCAL)

## 7 · Tokenize + pack the corpus
Uses the **original Llama 3 tokenizer** (configuration A of the tokenizer study). The printed `tokens_per_word (fertility)` and `chars_per_token` are your baseline numbers for that study.

In [ ]:
with open(CFG_PATH, "w") as f:
    json.dump(build_config(), f, indent=2)
rc = run_streaming([sys.executable, f"{SCRIPT_DIR}/prepare_data.py", "--config", CFG_PATH])
assert rc == 0, "data preparation failed"
STATS = json.load(open(DATA_DIR + "_stats.json"))

t = STATS["train"]
full_steps_per_epoch = t["blocks"] // (PER_DEVICE_BATCH * GRAD_ACCUM * N_GPUS)
print(f"\nFull corpus: {t['tokens']:,} train tokens -> {t['blocks']:,} blocks of {SEQ_LEN}")
print(f"Full run: ~{full_steps_per_epoch} optimizer steps/epoch x {NUM_EPOCHS} epochs "
      f"-> ~{full_steps_per_epoch * NUM_EPOCHS // SAVE_STEPS} checkpoints pushed to the Hub")

## 8 · Smoke test (`SMOKE_TEST = True`)
Phase A trains to step 5, pushes `checkpoint-5`, and stops as if it crashed. The local checkpoint folder is then deleted (like losing the Kaggle session). Phase B must restore `checkpoint-5` **from the Hub** and finish to step 10.

In [ ]:
if SMOKE_TEST:
    crash, last = SMOKE_CFG["crash_at_step"], SMOKE_CFG["max_steps"]
    print("Resetting the smoke repo and local folder for a clean test ...")
    api.delete_repo(REPO_ID, missing_ok=True)
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

    print(f"\n===== PHASE A: train -> checkpoint -> push -> simulated crash at step {crash} =====")
    sA = launch_training(crash_at_step=crash)

    print("\n===== Simulating a lost session: deleting all local checkpoints =====")
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

    print(f"\n===== PHASE B: resume from the Hub and finish at step {last} =====")
    sB = launch_training()

    files = hub_files(REPO_ID)
    rng_files = [f"rng_state_{r}.pth" for r in range(N_GPUS)] if N_GPUS > 1 else ["rng_state.pth"]
    checks = [
        ("Phase A stopped at the simulated crash",
         sA.get("stopped_reason") == "simulated_crash" and sA.get("end_step") == crash),
        (f"checkpoint-{crash}: adapter on Hub", f"checkpoint-{crash}/adapter_model.safetensors" in files),
        (f"checkpoint-{crash}: optimizer + scheduler + trainer state on Hub",
         all(f"checkpoint-{crash}/{x}" in files for x in ["optimizer.pt", "scheduler.pt", "trainer_state.json"])),
        (f"checkpoint-{crash}: RNG state of every GPU on Hub", all(f"checkpoint-{crash}/{x}" in files for x in rng_files)),
        ("Phase B resumed from that checkpoint", sB.get("start_step") == crash),
        (f"Phase B finished at step {last}", sB.get("finished") is True and sB.get("end_step") == last),
        (f"checkpoint-{last} on Hub", f"checkpoint-{last}/adapter_model.safetensors" in files),
        ("Final adapter + training_complete.json at repo root",
         "adapter_model.safetensors" in files and "training_complete.json" in files),
        ("Final eval loss is finite", math.isfinite(sB.get("final_eval", {}).get("eval_loss", float("nan")))),
    ]
    print("\n================ SMOKE TEST RESULTS ================")
    for name, ok in checks:
        print(("✅ " if ok else "❌ ") + name)

    hist = sA.get("eval_history", []) + sB.get("eval_history", [])
    if hist:
        print(f"\nEval perplexity (smoke subset): step {hist[0][0]}: {math.exp(hist[0][1]):.2f}  ->  "
              f"step {hist[-1][0]}: {math.exp(hist[-1][1]):.2f}")

    samples = (sA.get("tps_samples") or []) + (sB.get("tps_samples") or [])
    if samples:
        tps = statistics.median(samples)
        full_tokens = STATS["train"]["blocks"] * SEQ_LEN * NUM_EPOCHS
        hours = full_tokens / tps / 3600
        print(f"\nMeasured throughput ≈ {tps:,.0f} tokens/s on {N_GPUS} GPU(s)")
        print(f"Full run ({NUM_EPOCHS} epochs, {full_tokens:,} tokens) ≈ {hours:.1f} h of training "
              f"+ eval/checkpoint overhead ≈ {math.ceil(hours * 1.2 / MAX_RUNTIME_HOURS)} Kaggle session(s)")

    if not all(ok for _, ok in checks):
        raise RuntimeError("Smoke test FAILED - see the ❌ lines above.")
    print("\nAll checks passed. Set SMOKE_TEST = False and run the full training (Save & Run All).")

## 9 · Full training (`SMOKE_TEST = False`)
Re-running this notebook always continues from the newest checkpoint on the Hub.

In [ ]:
if not SMOKE_TEST:
    status = launch_training()
    if status.get("finished"):
        print(f"\n✅ Training complete: https://huggingface.co/{REPO_ID}")
    else:
        print(f"\n⏸ Stopped at step {status['end_step']}/{status['max_steps']} ({status['stopped_reason']}). "
              f"checkpoint-{status['end_step']} is on the Hub. Run the notebook again to continue.")

## 10 · Qualitative check: base Llama 3 vs. + SLTFM adapter

In [ ]:
if training_complete(REPO_ID):
    run_streaming([sys.executable, f"{SCRIPT_DIR}/generate.py", "--config", CFG_PATH, "--adapter", REPO_ID])
else:
    print("Training not finished yet - skipping. (To test an intermediate checkpoint add: '--subfolder', 'checkpoint-N')")

## Notes
- **Why QLoRA, not plain LoRA:** Llama 3 8B in fp16 is ~16 GB and a T4 has 15 GB, so each GPU holds a 4-bit NF4 copy and trains fp32 LoRA adapters (the plan's backup method). On an A100/H100 you can switch to bf16 LoRA.
- **Why fp16:** T4 has no bf16 support. Watch the loss in W&B; if you ever see `nan`, lower `LEARNING_RATE` to `1e-4`.
- **Hub repo layout:** `checkpoint-N/` for every save (adapter, optimizer, scheduler, RNG, trainer state), `wandb_run_id.txt` (keeps one W&B run across resumes), `sltfm_run_config.json`, and the final adapter + `training_complete.json` at the root.
- **Starting over:** delete the Hub repo (or change `HUB_REPO_NAME`). To resume with different hyper-parameters, start a new repo instead.
- **License:** Llama 3 derivatives must keep "Llama 3" in the model name and state "Built with Meta Llama 3"; the repo name and model card already do.